# The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power - Hypothesis Testing
notebook 02

Table of Content

- [Purpose and scope](#purpose-and-scope)
- [Load data and hypothesis mapping](#load-data-and-hypothesis-mapping)
- [Hypothesis testing plan](#hypothesis-testing-plan)
- [Methodological approach](#methodological-approach)
- [Pairwise testing sample diagnostics](#pairwise-testing-sample-diagnostics)
- [Hypothesis tests](#hypothesis-tests)
- [Results summary](#results-summary)
- [Interpretation and implications for the report](#interpretation-and-implications-for-the-report)

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests

## <a id="purpose-and-scope"></a>Purpose and scope


This notebook formally tests the working hypotheses defined for the project “The Gender Conversion Gap: From STEM and Digital Participation to Labour-Market Power.”

The analysis uses the OECD-member sample prepared in the previous notebooks and excludes aggregate areas such as OECD, EU27, G7, and EU28.

The term “conversion gap” is used as a project framing device, not as a literal cohort-level transition claim. The dataset does not follow the same women from STEM graduation into later management or pay outcomes. Instead, it tests whether recent country-level indicators of STEM and digital participation are associated with labour-market and leadership indicators across OECD countries.

The main analysis focuses on cross-country associations, not causal effects.

Primary hypotheses:

* H1: women’s STEM graduate share and women’s representation in senior/middle management;
* H2: women’s STEM graduate share and the median gender wage gap indicator.

Exploratory / secondary hypotheses:

* H3: youth coding gender gap and overall working-age labour-force participation gap;
* H4: part-time employment concentration gap and the median gender wage gap indicator.

H3 is treated as exploratory because it compares a youth digital activity indicator for ages 16–24 with an overall labour-force participation indicator for ages 15–74.

## <a id="load-data-and-hypothesis-mapping"></a>Load data and hypothesis mapping

In [2]:
DATA_PATH = '/Users/avr/Jupiter/Gender Inequality Report - startisctic capstone/data/processed/'

df = pd.read_csv(DATA_PATH + "gender_conversion_gap_latest.csv")
data_dict = pd.read_csv(DATA_PATH + "data_dictionary.csv")
hypotheses = pd.read_csv(DATA_PATH + "hypothesis_mapping.csv")

print("Main dataset:", df.shape)
display(df.head())

print("Hypothesis mapping:", hypotheses.shape)
display(hypotheses)

Main dataset: (60, 19)


,REF_AREA,is_oecd_member,is_aggregate_area,sample_scope,stem_women_share,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,gender_wage_gap_median,part_time_employment_gap_F_minus_M,women_senior_middle_mgmt_share,stem_women_share_year,coding_gap_youth_M_minus_F_year,labour_force_participation_gap_M_minus_F_year,gender_wage_gap_median_year,part_time_employment_gap_F_minus_M_year,women_senior_middle_mgmt_share_year,latest_year_min,latest_year_max,latest_year_span
0,ARG,False,False,non-member / partner / candidate,NaN,NaN,19.078,9.090909,26.545918,NaN,NaN,NaN,2023.0,2024.0,2019.0,NaN,2019.0,2024.0,5.0
1,AUS,True,False,OECD member,33.352506,NaN,7.650,10.671213,16.924777,39.39,2024.0,NaN,2025.0,2024.0,2025.0,2024.0,2024.0,2025.0,1.0
2,AUT,True,False,OECD member,29.190229,10.8682,8.144,11.062252,25.858781,33.79,2024.0,2025.0,2025.0,2024.0,2025.0,2023.0,2023.0,2025.0,2.0
3,BEL,True,False,OECD member,28.368823,9.9548,7.970,0.906909,18.302429,33.69,2024.0,2025.0,2025.0,2022.0,2025.0,2023.0,2022.0,2025.0,3.0
4,BGR,False,False,non-member / partner / candidate,36.155884,2.3700,9.686,2.360074,0.595541,NaN,2023.0,2025.0,2025.0,2024.0,2025.0,NaN,2023.0,2025.0,2.0


Hypothesis mapping: (4, 10)


,hypothesis_id,hypothesis_name,research_hypothesis,null_hypothesis,alternative_hypothesis,independent_variable,dependent_variable,expected_direction,sample_note,interpretation_note
0,H1,STEM-to-leadership conversion,Countries with higher shares of women among te...,There is no association between women’s share ...,There is an association between women’s share ...,stem_women_share,women_senior_middle_mgmt_share,positive,Compare all available observations with OECD-m...,Leadership is measured with a senior/middle ma...
1,H2,STEM-to-pay equality,Countries with higher shares of women among te...,There is no association between women’s share ...,There is an association between women’s share ...,stem_women_share,gender_wage_gap_median,negative,Compare all available observations with OECD-m...,"The wage gap is unadjusted, so results are des..."
2,H3,Digital-skills gap and labour-market participa...,Countries with larger male advantages in youth...,There is no association between the youth codi...,There is an association between the youth codi...,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,positive,Compare all available observations with OECD-m...,Interpret cautiously because coding is measure...
3,H4,Work-structure bottleneck,Countries where women are more concentrated in...,There is no association between the gender gap...,There is an association between the gender gap...,part_time_employment_gap_F_minus_M,gender_wage_gap_median,positive,Compare all available observations with OECD-m...,Part-time gap is a work-structure proxy and do...


In [3]:
# recreate analysis samples
# main sample for formal hypothesis testing:
# oecd member countries only, excluding aggregate areas
df_main = df[
    (df["is_oecd_member"]) &
    (~df["is_aggregate_area"])
].copy()

# broader non-aggregate sample for optional sensitivity checks
df_broader = df[
    ~df["is_aggregate_area"]
].copy()

print("Main OECD sample:", df_main.shape)
print("Broader non-aggregate sample:", df_broader.shape)

Main OECD sample: (38, 19)
Broader non-aggregate sample: (53, 19)


## <a id="hypothesis-testing-plan"></a>Hypothesis testing plan

The hypothesis testing plan follows the working hypotheses defined before formal testing. Expected directions are not changed based on preliminary correlation checks.

H1 and H2 are treated as the main primary hypotheses because they directly test the project’s central question: whether stronger women’s representation in the STEM education pipeline is associated with stronger labour-market power indicators.

H3 became exploratory / secondary hypothesis with an expected positive association and will be interpreted cautiously. This caution comes from the EDA/data diagnostics: the youth coding indicator refers to ages 16–24, while the labour-force participation indicator refers to ages 15–74, and this pair has the largest year-alignment caveat.

H4 is treated as a secondary hypothesis because it tests an alternative work-structure mechanism: whether women’s relative concentration in part-time employment is associated with the median gender wage gap indicator.

All hypotheses are tested as cross-country associations, not causal relationships.

In [4]:
hypothesis_plan = pd.DataFrame([
    {
        "hypothesis_id": "H1",
        "hypothesis_name": "STEM-to-leadership conversion",
        "status": "primary",
        "research_hypothesis": (
            "Countries with higher shares of women among tertiary STEM graduates "
            "tend to have higher shares of women in senior and middle management positions."
        ),
        "null_hypothesis": (
            "There is no association between women's share among tertiary STEM graduates "
            "and women's share in senior/middle management."
        ),
        "x": "stem_women_share",
        "y": "women_senior_middle_mgmt_share",
        "x_role": "predictor",
        "y_role": "outcome",
        "expected_direction": "positive",
        "test_idea": (
            "Pearson and Spearman correlation (two-sided); "
            "descriptive simple linear regression; influence diagnostics"
        ),
        "interpretation_caution": (
            "This is a cross-country association test, not a literal cohort-level "
            "transition from STEM graduation into management."
        ),
    },
    {
        "hypothesis_id": "H2",
        "hypothesis_name": "STEM-to-pay equality",
        "status": "primary",
        "research_hypothesis": (
            "Countries with higher shares of women among tertiary STEM graduates "
            "tend to have lower median gender wage gap indicator values."
        ),
        "null_hypothesis": (
            "There is no association between women's share among tertiary STEM graduates "
            "and the median gender wage gap indicator."
        ),
        "x": "stem_women_share",
        "y": "gender_wage_gap_median",
        "x_role": "predictor",
        "y_role": "outcome",
        "expected_direction": "negative",
        "test_idea": (
            "Pearson and Spearman correlation (two-sided); "
            "descriptive simple linear regression"
        ),
        "interpretation_caution": (
            "Higher median gender wage gap indicator values indicate worse pay outcomes for women."
        ),
    },
    {
        "hypothesis_id": "H3",
        "hypothesis_name": "Digital-skills gap and labour-market participation",
        "status": "secondary_exploratory",
        "research_hypothesis": (
            "Countries with larger male advantages in the youth coding indicator "
            "tend to have larger gender gaps in labour-force participation."
        ),
        "null_hypothesis": (
            "There is no association between the youth coding gender gap "
            "and the labour-force participation gender gap."
        ),
        "x": "coding_gap_youth_M_minus_F",
        "y": "labour_force_participation_gap_M_minus_F",
        "x_role": "predictor",
        "y_role": "outcome",
        "expected_direction": "positive",
        "test_idea": (
            "Pearson and Spearman correlation (two-sided); scatterplot; "
            "sensitivity checks excluding observations with pairwise year gap > 2 years "
            "and using exact-year matches where available"
        ),
        "interpretation_caution": (
            "This is exploratory because youth coding covers ages 16-24, while "
            "labour-force participation covers ages 15-74. This pair also has the "
            "largest year-alignment caveat in the dataset."
        ),
    },
    {
        "hypothesis_id": "H4",
        "hypothesis_name": "Work-structure bottleneck",
        "status": "secondary",
        "research_hypothesis": (
            "Countries where women are more concentrated in part-time employment relative to men "
            "tend to have larger median gender wage gap indicator values."
        ),
        "null_hypothesis": (
            "There is no association between the gender gap in part-time employment incidence "
            "and the median gender wage gap indicator."
        ),
        "x": "part_time_employment_gap_F_minus_M",
        "y": "gender_wage_gap_median",
        "x_role": "predictor",
        "y_role": "outcome",
        "expected_direction": "positive",
        "test_idea": (
            "Pearson and Spearman correlation (two-sided); "
            "descriptive simple linear regression"
        ),
        "interpretation_caution": (
            "Part-time employment incidence is a work-structure indicator, not a direct measure "
            "of care burden or discrimination."
        ),
    },
])

hypothesis_plan

,hypothesis_id,hypothesis_name,status,research_hypothesis,null_hypothesis,x,y,x_role,y_role,expected_direction,test_idea,interpretation_caution
0,H1,STEM-to-leadership conversion,primary,Countries with higher shares of women among te...,There is no association between women's share ...,stem_women_share,women_senior_middle_mgmt_share,predictor,outcome,positive,Pearson and Spearman correlation (two-sided); ...,"This is a cross-country association test, not ..."
1,H2,STEM-to-pay equality,primary,Countries with higher shares of women among te...,There is no association between women's share ...,stem_women_share,gender_wage_gap_median,predictor,outcome,negative,Pearson and Spearman correlation (two-sided); ...,Higher median gender wage gap indicator values...
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,Countries with larger male advantages in the y...,There is no association between the youth codi...,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,predictor,outcome,positive,Pearson and Spearman correlation (two-sided); ...,This is exploratory because youth coding cover...
3,H4,Work-structure bottleneck,secondary,Countries where women are more concentrated in...,There is no association between the gender gap...,part_time_employment_gap_F_minus_M,gender_wage_gap_median,predictor,outcome,positive,Pearson and Spearman correlation (two-sided); ...,Part-time employment incidence is a work-struc...


## <a id="methodological-approach"></a>Methodological approach

Each hypothesis is tested as a cross-country association between two continuous country-level indicators.

For each hypothesis pair, the notebook reports:

* pairwise complete sample size;
* pairwise year-gap diagnostics;
* Pearson correlation for linear association;
* Spearman correlation as a robustness check for monotonic association;
* simple OLS regression as a descriptive association model.

OLS results are interpreted descriptively: the slope estimates the average country-level change in the outcome associated with a one-unit increase in the predictor. These results should not be interpreted causally.

The main analysis uses OECD member countries only. Broader non-aggregate samples may be used only as sensitivity checks.

### Multiple-testing policy

H1 and H2 form the pre-specified primary hypothesis family. Their primary
Pearson correlation p-values are adjusted using the Holm procedure across these
two tests.

Spearman correlations are reported as robustness checks. In a bivariate OLS
model with one predictor, the slope test is mathematically aligned with the
Pearson correlation test. Pearson is therefore reported as the primary
linear-association statistic, while OLS is reported to express the estimated
association in the original indicator units, with a confidence interval and
R-squared.

H3, H4, and H3 sensitivity analyses are exploratory. Their two-sided raw
p-values are reported alongside effect sizes and confidence intervals and are
not used for confirmatory claims.


## <a id="pairwise-testing-sample-diagnostics"></a>Pairwise testing sample diagnostics

### Main association tests

The following tests use pairwise complete OECD-member observations for each hypothesis.

All correlation tests are reported as two-sided tests. Expected directions are used for interpretation, not for changing the statistical test after seeing the data.

Simple OLS regression is used as a descriptive association model. The slope should be interpreted as the average country-level difference in the outcome associated with a one-unit difference in the predictor, not as a causal effect.

In [5]:
def pairwise_sample_diagnostics(df_sample, plan):
    rows = []

    for _, h in plan.iterrows():
        x = h["x"]
        y = h["y"]
        x_year = f"{x}_year"
        y_year = f"{y}_year"

        clean = df_sample[["REF_AREA", x, y, x_year, y_year]].dropna().copy()
        clean["pairwise_year_gap"] = (clean[x_year] - clean[y_year]).abs()

        max_gap = clean["pairwise_year_gap"].max()
        median_gap = clean["pairwise_year_gap"].median()

        largest_gap_countries = (
            clean.sort_values(["pairwise_year_gap", "REF_AREA"], ascending=[False, True])
            .head(3)
            [["REF_AREA", x_year, y_year, "pairwise_year_gap"]]
        )

        rows.append({
            "hypothesis_id": h["hypothesis_id"],
            "hypothesis_name": h["hypothesis_name"],
            "status": h["status"],
            "x": x,
            "y": y,
            "n_pairwise_complete": len(clean),
            "estonia_included": "EST" in clean["REF_AREA"].values,
            "median_pairwise_year_gap": median_gap,
            "max_pairwise_year_gap": max_gap,
            "largest_gap_countries": "; ".join(
                f"{r.REF_AREA} ({int(r.pairwise_year_gap)}y)"
                for _, r in largest_gap_countries.iterrows()
            ),
        })

    return pd.DataFrame(rows)

pairwise_testing_samples = pairwise_sample_diagnostics(df_main, hypothesis_plan)
pairwise_testing_samples

,hypothesis_id,hypothesis_name,status,x,y,n_pairwise_complete,estonia_included,median_pairwise_year_gap,max_pairwise_year_gap,largest_gap_countries
0,H1,STEM-to-leadership conversion,primary,stem_women_share,women_senior_middle_mgmt_share,30,True,1.0,1.0,AUT (1y); BEL (1y); CHE (1y)
1,H2,STEM-to-pay equality,primary,stem_women_share,gender_wage_gap_median,38,True,0.0,2.0,BEL (2y); GRC (2y); ISR (2y)
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,32,True,0.0,8.0,CHL (8y); GBR (6y); ISL (4y)
3,H4,Work-structure bottleneck,secondary,part_time_employment_gap_F_minus_M,gender_wage_gap_median,36,True,1.0,3.0,BEL (3y); GRC (3y); ISL (3y)


In [6]:
pairwise_testing_samples.to_csv(
    DATA_PATH + "pairwise_testing_samples_oecd.csv",
    index=False
)

In [7]:
# helper for pairwise data
def get_pairwise_data(df_sample, x, y):
    x_year = f"{x}_year"
    y_year = f"{y}_year"

    cols = ["REF_AREA", x, y]

    if x_year in df_sample.columns and y_year in df_sample.columns:
        cols += [x_year, y_year]

    pair_df = df_sample[cols].dropna().copy()

    if x_year in pair_df.columns and y_year in pair_df.columns:
        pair_df["pairwise_year_gap"] = (pair_df[x_year] - pair_df[y_year]).abs()

    return pair_df

In [8]:
# run test for one pair
def run_association_test(df_sample, hypothesis_row):
    h_id = hypothesis_row["hypothesis_id"]
    x = hypothesis_row["x"]
    y = hypothesis_row["y"]
    expected_direction = hypothesis_row["expected_direction"]

    pair_df = get_pairwise_data(df_sample, x, y)

    n = len(pair_df)

    if n < 3:
        return {
            "hypothesis_id": h_id,
            "x": x,
            "y": y,
            "n": n,
            "error": "Not enough observations",
        }

    # Pearson correlation
    pearson_r, pearson_p = stats.pearsonr(pair_df[x], pair_df[y])

    # Spearman correlation
    spearman_rho, spearman_p = stats.spearmanr(pair_df[x], pair_df[y])

    # Simple OLS regression: y ~ x
    X = sm.add_constant(pair_df[x])
    model = sm.OLS(pair_df[y], X).fit()

    slope = model.params[x]
    intercept = model.params["const"]
    p_value_slope = model.pvalues[x]
    r_squared = model.rsquared

    ci_low, ci_high = model.conf_int().loc[x]

    if expected_direction == "positive":
        direction_match = slope > 0
    elif expected_direction == "negative":
        direction_match = slope < 0
    else:
        direction_match = np.nan

    result = {
        "hypothesis_id": h_id,
        "hypothesis_name": hypothesis_row["hypothesis_name"],
        "status": hypothesis_row["status"],
        "x": x,
        "y": y,
        "expected_direction": expected_direction,
        "n": n,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "ols_intercept": intercept,
        "ols_slope": slope,
        "ols_slope_ci_low": ci_low,
        "ols_slope_ci_high": ci_high,
        "ols_slope_p": p_value_slope,
        "ols_r_squared": r_squared,
        "direction_matches_expectation": direction_match,
        "estonia_included": "EST" in pair_df["REF_AREA"].values,
    }

    if "pairwise_year_gap" in pair_df.columns:
        result["median_pairwise_year_gap"] = pair_df["pairwise_year_gap"].median()
        result["max_pairwise_year_gap"] = pair_df["pairwise_year_gap"].max()

    return result

In [9]:
# run all H1–H4 tests
test_results = []

for _, h in hypothesis_plan.iterrows():
    result = run_association_test(df_main, h)
    test_results.append(result)

hypothesis_test_results = pd.DataFrame(test_results)

# H1 and H2 form the pre-specified primary hypothesis family.
# Holm adjustment applies only to their primary Pearson correlation p-values.
hypothesis_test_results["pearson_p_holm_primary"] = np.nan
hypothesis_test_results["pearson_holm_reject_primary"] = False

primary_mask = hypothesis_test_results["status"] == "primary"
primary_p_values = hypothesis_test_results.loc[primary_mask, "pearson_p"]

primary_reject, primary_p_adjusted, _, _ = multipletests(
    primary_p_values,
    alpha=0.05,
    method="holm",
)

hypothesis_test_results.loc[
    primary_mask,
    "pearson_p_holm_primary",
] = primary_p_adjusted
hypothesis_test_results.loc[
    primary_mask,
    "pearson_holm_reject_primary",
] = primary_reject

hypothesis_test_results

,hypothesis_id,hypothesis_name,status,x,y,expected_direction,n,pearson_r,pearson_p,spearman_rho,...,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,estonia_included,median_pairwise_year_gap,max_pairwise_year_gap,pearson_p_holm_primary,pearson_holm_reject_primary
0,H1,STEM-to-leadership conversion,primary,stem_women_share,women_senior_middle_mgmt_share,positive,30,0.426925,0.018629,0.247164,...,0.101498,1.026006,0.018629,0.182265,True,True,1.0,1.0,0.037257,True
1,H2,STEM-to-pay equality,primary,stem_women_share,gender_wage_gap_median,negative,38,-0.174451,0.294857,-0.121348,...,-0.614348,0.191808,0.294857,0.030433,True,True,0.0,2.0,0.294857,False
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,positive,32,-0.515065,0.002557,-0.460044,...,-0.999795,-0.234132,0.002557,0.265292,False,True,0.0,8.0,NaN,False
3,H4,Work-structure bottleneck,secondary,part_time_employment_gap_F_minus_M,gender_wage_gap_median,positive,36,-0.214115,0.209858,-0.212098,...,-0.364185,0.082961,0.209858,0.045845,False,True,1.0,3.0,NaN,False


In [10]:
# rounded
hypothesis_test_results_rounded = hypothesis_test_results.copy()

round_cols = [
    "pearson_r",
    "pearson_p",
    "pearson_p_holm_primary",
    "spearman_rho",
    "spearman_p",
    "ols_intercept",
    "ols_slope",
    "ols_slope_ci_low",
    "ols_slope_ci_high",
    "ols_slope_p",
    "ols_r_squared",
    "median_pairwise_year_gap",
    "max_pairwise_year_gap",
]

existing_round_cols = [
    col for col in round_cols
    if col in hypothesis_test_results_rounded.columns
]

hypothesis_test_results_rounded[existing_round_cols] = (
    hypothesis_test_results_rounded[existing_round_cols]
    .round(4)
)

hypothesis_test_results_rounded

,hypothesis_id,hypothesis_name,status,x,y,expected_direction,n,pearson_r,pearson_p,spearman_rho,...,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,estonia_included,median_pairwise_year_gap,max_pairwise_year_gap,pearson_p_holm_primary,pearson_holm_reject_primary
0,H1,STEM-to-leadership conversion,primary,stem_women_share,women_senior_middle_mgmt_share,positive,30,0.4269,0.0186,0.2472,...,0.1015,1.0260,0.0186,0.1823,True,True,1.0,1.0,0.0373,True
1,H2,STEM-to-pay equality,primary,stem_women_share,gender_wage_gap_median,negative,38,-0.1745,0.2949,-0.1213,...,-0.6143,0.1918,0.2949,0.0304,True,True,0.0,2.0,0.2949,False
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,positive,32,-0.5151,0.0026,-0.4600,...,-0.9998,-0.2341,0.0026,0.2653,False,True,0.0,8.0,NaN,False
3,H4,Work-structure bottleneck,secondary,part_time_employment_gap_F_minus_M,gender_wage_gap_median,positive,36,-0.2141,0.2099,-0.2121,...,-0.3642,0.0830,0.2099,0.0458,False,True,1.0,3.0,NaN,False


In [11]:
hypothesis_test_results_rounded.to_csv(
    DATA_PATH + "hypothesis_test_results_oecd.csv",
    index=False
)

## <a id="hypothesis-tests"></a>Hypothesis tests

In [12]:
# visual for each H
def plot_hypothesis_scatter(df_sample, hypothesis_row):
    h_id = hypothesis_row["hypothesis_id"]
    h_name = hypothesis_row["hypothesis_name"]
    x = hypothesis_row["x"]
    y = hypothesis_row["y"]
    expected_direction = hypothesis_row["expected_direction"]

    plot_df = get_pairwise_data(df_sample, x, y).copy()

    plot_df["country_group"] = np.where(
        plot_df["REF_AREA"] == "EST",
        "Estonia",
        "Other OECD countries"
    )

    fig = px.scatter(
        plot_df,
        x=x,
        y=y,
        color="country_group",
        hover_name="REF_AREA",
        trendline="ols",
        title=f"{h_id}: {h_name}",
        labels={
            x: x,
            y: y,
            "country_group": "Country group",
        },
    )

    fig.update_layout(
        height=550,
        legend_title_text="Country group",
    )

    fig.add_annotation(
        text=f"Expected direction: {expected_direction}",
        xref="paper",
        yref="paper",
        x=0.01,
        y=0.99,
        showarrow=False,
        align="left",
        bgcolor="rgba(255,255,255,0.7)",
    )

    fig.show()

In [13]:
# helper to get result
def show_hypothesis_result(results_df, hypothesis_id):
    cols = [
        "hypothesis_id",
        "hypothesis_name",
        "status",
        "expected_direction",
        "n",
        "pearson_r",
        "pearson_p",
        "pearson_p_holm_primary",
        "pearson_holm_reject_primary",
        "spearman_rho",
        "spearman_p",
        "ols_slope",
        "ols_slope_ci_low",
        "ols_slope_ci_high",
        "ols_slope_p",
        "ols_r_squared",
        "direction_matches_expectation",
        "median_pairwise_year_gap",
        "max_pairwise_year_gap",
    ]

    existing_cols = [col for col in cols if col in results_df.columns]

    return results_df.loc[
        results_df["hypothesis_id"] == hypothesis_id,
        existing_cols
    ]

### H1 — STEM-to-leadership conversion

In [14]:
h1 = hypothesis_plan[hypothesis_plan["hypothesis_id"] == "H1"].iloc[0]

plot_hypothesis_scatter(df_main, h1)
show_hypothesis_result(hypothesis_test_results_rounded, "H1")

,hypothesis_id,hypothesis_name,status,expected_direction,n,pearson_r,pearson_p,pearson_p_holm_primary,pearson_holm_reject_primary,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,median_pairwise_year_gap,max_pairwise_year_gap
0,H1,STEM-to-leadership conversion,primary,positive,30,0.4269,0.0186,0.0373,True,0.2472,0.1879,0.5638,0.1015,1.026,0.0186,0.1823,True,1.0,1.0


### H1 presentation visual: Estonia and Japan highlighted

This version keeps the full OECD sample and overall OLS line visible while
highlighting Estonia and Japan as two labelled country cases.


In [15]:
# H1 presentation visual: Estonia and Japan highlighted

h1_presentation = hypothesis_plan[
    hypothesis_plan["hypothesis_id"] == "H1"
].iloc[0]

h1_plot_df = get_pairwise_data(
    df_main,
    h1_presentation["x"],
    h1_presentation["y"],
).copy()

x = h1_presentation["x"]
y = h1_presentation["y"]

other_countries = h1_plot_df[
    ~h1_plot_df["REF_AREA"].isin(["EST", "JPN"])
].copy()

fig = px.scatter(
    other_countries,
    x=x,
    y=y,
    hover_name="REF_AREA",
    title="H1: Women in STEM and senior/middle management",
    labels={
        x: "Women among tertiary STEM graduates (%)",
        y: "Women in senior/middle management (%)",
    },
    color_discrete_sequence=["#999999"],
)

fig.update_traces(
    name="Other OECD countries",
    showlegend=True,
    marker=dict(size=9, opacity=0.7),
)

highlight_specs = [
    ("EST", "Estonia", "#0072B2", "top right"),
    ("JPN", "Japan", "#D55E00", "bottom left"),
]

for country_code, country_name, color, text_position in highlight_specs:
    row = h1_plot_df[h1_plot_df["REF_AREA"] == country_code].iloc[0]

    fig.add_scatter(
        x=[row[x]],
        y=[row[y]],
        mode="markers+text",
        text=[country_name],
        textposition=text_position,
        hovertemplate=(
            f"{country_name}<br>"
            "Women among tertiary STEM graduates: %{x:.1f}%<br>"
            "Women in senior/middle management: %{y:.1f}%<extra></extra>"
        ),
        marker=dict(color=color, size=13, line=dict(color="#333333", width=1)),
        name=country_name,
    )

ols_slope, ols_intercept = np.polyfit(h1_plot_df[x], h1_plot_df[y], 1)
x_line = np.linspace(h1_plot_df[x].min(), h1_plot_df[x].max(), 100)

fig.add_scatter(
    x=x_line,
    y=ols_intercept + ols_slope * x_line,
    mode="lines",
    line=dict(color="#333333", width=2),
    name="OLS line (all OECD countries)",
)

fig.update_layout(
    height=580,
    width=1100,
    legend_title_text="",
    plot_bgcolor="white",
)

fig.show()


In [16]:
# influense check
def ols_influence_diagnostics(df_sample, hypothesis_row):
    x = hypothesis_row["x"]
    y = hypothesis_row["y"]

    pair_df = get_pairwise_data(df_sample, x, y).copy()

    X = sm.add_constant(pair_df[x])
    model = sm.OLS(pair_df[y], X).fit()

    influence = model.get_influence()
    cooks_d = influence.cooks_distance[0]
    leverage = influence.hat_matrix_diag

    influence_df = pair_df[["REF_AREA", x, y]].copy()
    influence_df["cooks_d"] = cooks_d
    influence_df["leverage"] = leverage
    influence_df["studentized_residual"] = influence.resid_studentized_external

    influence_df = influence_df.sort_values("cooks_d", ascending=False)

    return influence_df

h1_influence = ols_influence_diagnostics(df_main, h1)
h1_influence.head(10)

,REF_AREA,stem_women_share,women_senior_middle_mgmt_share,cooks_d,leverage,studentized_residual
35,JPN,18.078102,13.27,1.210606,0.328852,-2.405421
34,ITA,39.539780,24.82,0.153356,0.072664,-2.094729
38,LUX,32.162662,16.98,0.120028,0.036549,-2.807777
26,GRC,41.682659,31.36,0.079967,0.107534,-1.159157
16,EST,41.545704,31.51,0.072735,0.104977,-1.118659
28,HUN,28.672234,38.47,0.046833,0.064795,1.170364
58,USA,38.482995,44.33,0.046429,0.059508,1.222063
15,ESP,27.498438,36.67,0.043179,0.080837,0.990603
55,SWE,38.844210,43.80,0.041014,0.063704,1.102204
39,LVA,33.467594,41.57,0.025161,0.033468,1.215762


In [17]:
# leave one out check
def leave_one_out_correlations(df_sample, hypothesis_row):
    x = hypothesis_row["x"]
    y = hypothesis_row["y"]

    pair_df = get_pairwise_data(df_sample, x, y).copy()

    rows = []

    for country in pair_df["REF_AREA"]:
        loo_df = pair_df[pair_df["REF_AREA"] != country].copy()

        pearson_r, pearson_p = stats.pearsonr(loo_df[x], loo_df[y])
        spearman_rho, spearman_p = stats.spearmanr(loo_df[x], loo_df[y])

        rows.append({
            "excluded_country": country,
            "n": len(loo_df),
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
        })

    return pd.DataFrame(rows).sort_values("pearson_r")

h1_leave_one_out = leave_one_out_correlations(df_main, h1)
h1_leave_one_out.head(10)

,excluded_country,n,pearson_r,pearson_p,spearman_rho,spearman_p
18,JPN,29,0.174801,0.364440,0.166502,0.387999
24,POL,29,0.399017,0.032016,0.196059,0.308059
28,SWE,29,0.400723,0.031222,0.208374,0.278040
29,USA,29,0.402494,0.030414,0.208374,0.278040
5,DEU,29,0.406459,0.028668,0.204926,0.286249
25,PRT,29,0.422532,0.022404,0.243842,0.202410
14,IRL,29,0.422668,0.022356,0.235961,0.217843
15,ISL,29,0.423181,0.022177,0.225616,0.239282
11,GBR,29,0.424635,0.021675,0.233005,0.223831
23,NOR,29,0.426278,0.021119,0.242365,0.205245


In [18]:
h1_leave_one_out.tail(10)

,excluded_country,n,pearson_r,pearson_p,spearman_rho,spearman_p
0,AUS,29,0.433695,0.018752,0.254680,0.182442
16,ISR,29,0.433983,0.018664,0.285222,0.133678
4,CZE,29,0.437069,0.017750,0.247783,0.194982
21,LVA,29,0.438266,0.017405,0.253695,0.184198
20,LUX,29,0.450473,0.014196,0.211823,0.269984
7,ESP,29,0.454975,0.013144,0.259606,0.173837
13,HUN,29,0.458996,0.012260,0.285222,0.133678
8,EST,29,0.463651,0.011300,0.322660,0.087795
12,GRC,29,0.465934,0.010852,0.322660,0.087795
17,ITA,29,0.501642,0.005563,0.331527,0.078940


### H1 — STEM-to-leadership conversion

H1 tested whether countries with higher shares of women among tertiary STEM graduates tend to have higher shares of women in senior and middle management positions.

The primary Pearson correlation indicates a positive linear association, r = 0.43, raw p = 0.019, with 30 OECD countries included in the pairwise complete sample. This result remains below 0.05 after Holm adjustment across the two pre-specified primary Pearson tests (adjusted p = 0.037).

The corresponding bivariate OLS model is reported to express the association in indicator units: a one percentage-point higher women’s STEM graduate share is associated with an estimated 0.56 percentage-point higher women’s share in senior/middle management. The 95% confidence interval for this estimated slope ranges from 0.10 to 1.03, and the OLS R² is 0.18.

The Spearman correlation is weaker, rho = 0.25, suggesting that the association may not be strongly monotonic across the full ranking of countries.

Overall, H1 receives partial support: there is evidence of a positive country-level linear association, but the relationship should be interpreted cautiously and checked for influential observations. This result should not be interpreted as a literal cohort-level transition from recent STEM graduation into management.

### H1 influence and leave-one-out diagnostics

Because the Pearson and Spearman results for H1 differ noticeably, influence diagnostics were used to assess whether the positive Pearson/OLS result is sensitive to individual countries.

Japan has the highest Cook’s distance by a large margin. It combines a very low women’s STEM graduate share with a very low women’s senior/middle management share, which gives it high leverage in the simple linear regression. Italy and Luxembourg also appear as notable observations, but their influence is much smaller than Japan’s.

The leave-one-out check confirms that the H1 result is sensitive to Japan. With all 30 OECD countries included, Pearson r is 0.43. When Japan is excluded, Pearson r drops to 0.17 and is no longer statistically significant. This indicates that the positive linear association is partly driven by Japan’s low-low position.

Therefore, H1 receives only partial support. The main Pearson/OLS result is positive, but it should not be treated as a robust general OECD-wide pattern.

### H2 — STEM-to-pay equality

In [19]:
h2 = hypothesis_plan[hypothesis_plan["hypothesis_id"] == "H2"].iloc[0]

plot_hypothesis_scatter(df_main, h2)
show_hypothesis_result(hypothesis_test_results_rounded, "H2")

,hypothesis_id,hypothesis_name,status,expected_direction,n,pearson_r,pearson_p,pearson_p_holm_primary,pearson_holm_reject_primary,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,median_pairwise_year_gap,max_pairwise_year_gap
1,H2,STEM-to-pay equality,primary,negative,38,-0.1745,0.2949,0.2949,False,-0.1213,0.468,-0.2113,-0.6143,0.1918,0.2949,0.0304,True,0.0,2.0


### H2 — STEM-to-pay equality

H2 tested whether countries with higher shares of women among tertiary STEM graduates tend to have lower median gender wage gap indicator values.

The estimated association is in the expected negative direction, but it is weak and not statistically significant. Pearson correlation is r = -0.17, p = 0.295, and Spearman correlation is rho = -0.12. The OLS slope is -0.21, with a confidence interval crossing zero, from -0.61 to 0.19. The model explains only about 3% of cross-country variation in the median gender wage gap indicator.

This means that the analysis does not provide strong evidence that countries with higher women’s STEM graduate shares tend to have lower median gender wage gap indicator values.

Substantively, this is still an important result for the project. It suggests that stronger representation in the STEM education pipeline, by itself, may not be a reliable country-level marker of pay equality. This supports the need to examine labour-market structure, leadership representation, and other institutional factors rather than treating STEM representation as a direct proxy for pay equality.


### H3 — Digital-skills gap and labour-market participation

In [20]:
h3 = hypothesis_plan[hypothesis_plan["hypothesis_id"] == "H3"].iloc[0]

plot_hypothesis_scatter(df_main, h3)
show_hypothesis_result(hypothesis_test_results_rounded, "H3")

,hypothesis_id,hypothesis_name,status,expected_direction,n,pearson_r,pearson_p,pearson_p_holm_primary,pearson_holm_reject_primary,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,median_pairwise_year_gap,max_pairwise_year_gap
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,positive,32,-0.5151,0.0026,NaN,False,-0.46,0.0081,-0.617,-0.9998,-0.2341,0.0026,0.2653,False,0.0,8.0


In [21]:
# H3 sensitivity checks
# association test on prepared pairwise data

def run_association_test_on_pair_df(pair_df, hypothesis_row, sample_label):
    x = hypothesis_row["x"]
    y = hypothesis_row["y"]
    expected_direction = hypothesis_row["expected_direction"]

    n = len(pair_df)

    if n < 3:
        return {
            "hypothesis_id": hypothesis_row["hypothesis_id"],
            "sample_label": sample_label,
            "x": x,
            "y": y,
            "n": n,
            "note": "Not enough observations for correlation/regression",
        }

    pearson_r, pearson_p = stats.pearsonr(pair_df[x], pair_df[y])
    spearman_rho, spearman_p = stats.spearmanr(pair_df[x], pair_df[y])

    X = sm.add_constant(pair_df[x])
    model = sm.OLS(pair_df[y], X).fit()

    slope = model.params[x]
    ci_low, ci_high = model.conf_int().loc[x]

    if expected_direction == "positive":
        direction_match = slope > 0
    elif expected_direction == "negative":
        direction_match = slope < 0
    else:
        direction_match = np.nan

    result = {
        "hypothesis_id": hypothesis_row["hypothesis_id"],
        "sample_label": sample_label,
        "x": x,
        "y": y,
        "n": n,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_rho,
        "spearman_p": spearman_p,
        "ols_slope": slope,
        "ols_slope_ci_low": ci_low,
        "ols_slope_ci_high": ci_high,
        "ols_slope_p": model.pvalues[x],
        "ols_r_squared": model.rsquared,
        "direction_matches_expectation": direction_match,
        "estonia_included": "EST" in pair_df["REF_AREA"].values,
    }

    if "pairwise_year_gap" in pair_df.columns:
        result["median_pairwise_year_gap"] = pair_df["pairwise_year_gap"].median()
        result["max_pairwise_year_gap"] = pair_df["pairwise_year_gap"].max()

    return result

# build H3 sensitivity samples
h3 = hypothesis_plan[hypothesis_plan["hypothesis_id"] == "H3"].iloc[0]

h3_pair = get_pairwise_data(df_main, h3["x"], h3["y"]).copy()

h3_main = h3_pair.copy()

h3_year_gap_le_2 = h3_pair[
    h3_pair["pairwise_year_gap"] <= 2
].copy()

h3_exact_year = h3_pair[
    h3_pair["pairwise_year_gap"] == 0
].copy()

print("H3 main sample:", h3_main.shape)
print("H3 year gap <= 2 sample:", h3_year_gap_le_2.shape)
print("H3 exact-year sample:", h3_exact_year.shape)

H3 main sample: (32, 6)
H3 year gap <= 2 sample: (28, 6)
H3 exact-year sample: (25, 6)


In [22]:
# which countries are excluded by year-gap filter?
h3_excluded_year_gap_gt_2 = h3_pair[
    h3_pair["pairwise_year_gap"] > 2
].sort_values("pairwise_year_gap", ascending=False)

h3_excluded_year_gap_gt_2[
    [
        "REF_AREA",
        h3["x"],
        h3["y"],
        f"{h3['x']}_year",
        f"{h3['y']}_year",
        "pairwise_year_gap",
    ]
]

,REF_AREA,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,coding_gap_youth_M_minus_F_year,labour_force_participation_gap_M_minus_F_year,pairwise_year_gap
8,CHL,3.919409,17.976,2017.0,2025.0,8.0
25,GBR,19.034600,7.543,2019.0,2025.0,6.0
32,ISL,16.237400,5.471,2021.0,2025.0,4.0
6,CAN,16.860000,7.674,2022.0,2025.0,3.0


In [23]:
# run h3 sensitivity tests
h3_sensitivity_results = pd.DataFrame([
    run_association_test_on_pair_df(
        h3_main,
        h3,
        "main_pairwise_complete"
    ),
    run_association_test_on_pair_df(
        h3_year_gap_le_2,
        h3,
        "year_gap_le_2"
    ),
    run_association_test_on_pair_df(
        h3_exact_year,
        h3,
        "exact_year_only"
    ),
])

round_cols = [
    "pearson_r",
    "pearson_p",
    "spearman_rho",
    "spearman_p",
    "ols_slope",
    "ols_slope_ci_low",
    "ols_slope_ci_high",
    "ols_slope_p",
    "ols_r_squared",
    "median_pairwise_year_gap",
    "max_pairwise_year_gap",
]

existing_round_cols = [
    col for col in round_cols
    if col in h3_sensitivity_results.columns
]

h3_sensitivity_results[existing_round_cols] = (
    h3_sensitivity_results[existing_round_cols]
    .round(4)
)

h3_sensitivity_results

,hypothesis_id,sample_label,x,y,n,pearson_r,pearson_p,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,estonia_included,median_pairwise_year_gap,max_pairwise_year_gap
0,H3,main_pairwise_complete,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,32,-0.5151,0.0026,-0.4600,0.0081,-0.6170,-0.9998,-0.2341,0.0026,0.2653,False,True,0.0,8.0
1,H3,year_gap_le_2,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,28,-0.4796,0.0098,-0.4116,0.0295,-0.6058,-1.0525,-0.1590,0.0098,0.2300,False,True,0.0,1.0
2,H3,exact_year_only,coding_gap_youth_M_minus_F,labour_force_participation_gap_M_minus_F,25,-0.3619,0.0754,-0.1815,0.3851,-0.3858,-0.8145,0.0428,0.0754,0.1310,False,True,0.0,0.0


In [24]:
fig = px.scatter(
    h3_year_gap_le_2,
    x=h3["x"],
    y=h3["y"],
    hover_name="REF_AREA",
    color=np.where(
        h3_year_gap_le_2["REF_AREA"] == "EST",
        "Estonia",
        "Other OECD countries"
    ),
    trendline="ols",
    title="H3 sensitivity: year gap <= 2 years",
)

fig.update_layout(
    height=550,
    legend_title_text="Country group",
)

fig.show()

### H3 — Digital-skills gap and labour-market participation

H3 tested whether countries with larger male advantages in the youth coding indicator tend to have larger gender gaps in labour-force participation.

The observed association is statistically significant but opposite to the expected direction. Pearson correlation is r = -0.52, p = 0.003, and Spearman correlation is rho = -0.46. The OLS slope is -0.62, with a confidence interval from -1.00 to -0.23. This means that, in this OECD sample, larger male advantages in youth coding are associated with smaller labour-force participation gaps, not larger ones.

Because this hypothesis was already classified as secondary/exploratory based on EDA diagnostics, this result should be interpreted cautiously. The two indicators measure different populations: youth coding covers ages 16–24, while labour-force participation covers ages 15–74. This pair also has the largest year-alignment caveat, with a maximum pairwise year gap of 8 years.

Therefore, H3 does not support the expected positive association. Instead, it suggests that the relationship between youth digital participation gaps and overall labour-market participation gaps is more complex than the original hypothesis assumed.

### H3 sensitivity check: pairwise year alignment

H3 compares the youth coding gender gap with the overall labour-force participation gender gap. This pair has the largest year-alignment caveat in the dataset, so an additional sensitivity check was conducted.

In the main pairwise complete OECD sample, the association is negative and statistically significant: Pearson r = -0.52 and Spearman rho = -0.46. This is opposite to the expected positive direction.

To check whether this result is driven by countries with large year gaps between the two indicators, the analysis was repeated after excluding observations with pairwise year gaps greater than 2 years. This removed Chile, the United Kingdom, Iceland, and Canada. The negative association remained: Pearson r = -0.48 and Spearman rho = -0.41.

An exact-year subset was also tested. In this smaller sample, the relationship remained negative in direction but became weaker and was no longer statistically significant.

Overall, the H3 sensitivity checks show that the negative association is not driven only by countries with large year gaps. However, because the exact-year subset is weaker and because the two indicators refer to different age groups, H3 should be interpreted as an exploratory result. The expected positive association is not supported; instead, the results suggest that youth coding gender gaps and overall labour-force participation gaps may reflect different country-level mechanisms.


### H4 — Work-structure bottleneck

In [25]:
h4 = hypothesis_plan[hypothesis_plan["hypothesis_id"] == "H4"].iloc[0]

plot_hypothesis_scatter(df_main, h4)
show_hypothesis_result(hypothesis_test_results_rounded, "H4")

,hypothesis_id,hypothesis_name,status,expected_direction,n,pearson_r,pearson_p,pearson_p_holm_primary,pearson_holm_reject_primary,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,median_pairwise_year_gap,max_pairwise_year_gap
3,H4,Work-structure bottleneck,secondary,positive,36,-0.2141,0.2099,NaN,False,-0.2121,0.2143,-0.1406,-0.3642,0.083,0.2099,0.0458,False,1.0,3.0


### H4 — Work-structure bottleneck

H4 tested whether countries where women are more concentrated in part-time employment relative to men tend to have larger median gender wage gap indicator values.

The observed association is weak, negative, and not statistically significant. Pearson correlation is r = -0.21, p = 0.210, and Spearman correlation is rho = -0.21. The OLS slope is -0.14, with a confidence interval crossing zero, from -0.36 to 0.08. The model explains about 5% of cross-country variation in the median gender wage gap indicator.

This result does not support the expected positive association. In this OECD cross-section, the selected gender gap in part-time employment incidence is not strongly associated with the selected median gender wage gap indicator in a simple bivariate analysis.

This result does not establish that part-time employment is irrelevant to pay inequality. It only shows that this particular country-level part-time gap does not provide a strong standalone statistical marker of the wage-gap indicator in the present dataset. The analysis does not measure care burden, occupational segregation, wage-setting institutions, parental leave arrangements, or career progression, so it cannot test whether such factors account for the observed cross-country variation.

In [26]:
# save per-hypothesis pairwise data

for _, h in hypothesis_plan.iterrows():
    h_id = h["hypothesis_id"].lower()
    pair_df = get_pairwise_data(df_main, h["x"], h["y"])

    pair_df.to_csv(
        DATA_PATH + f"{h_id}_pairwise_oecd_sample.csv",
        index=False
    )

## <a id="results-summary"></a>Results summary

In [27]:
result_interpretations = {
    "H1": {
        "result_assessment": "partial support",
        "short_result": (
            "Positive linear association in the main sample, but weak Spearman result and strong "
            "sensitivity to Japan in influence diagnostics."
        ),
        "interpretation_for_report": (
            "There is some evidence of a positive country-level linear association between "
            "women's STEM graduate share and women's representation in senior/middle management, "
            "but the pattern is not robust enough to describe as a general OECD-wide relationship."
        ),
    },
    "H2": {
        "result_assessment": "not supported / inconclusive",
        "short_result": (
            "Estimated association is weakly negative, as expected, but not statistically significant."
        ),
        "interpretation_for_report": (
            "Higher women's STEM graduate share is not strongly associated with lower median gender wage gap "
            "indicator values across OECD countries."
        ),
    },
    "H3": {
        "result_assessment": "expected direction not supported",
        "short_result": (
            "Association is negative and statistically significant in the main sample; remains negative "
            "after excluding countries with large pairwise year gaps, but weakens in exact-year subset."
        ),
        "interpretation_for_report": (
            "Youth coding gender gaps and overall labour-force participation gaps do not align in the "
            "expected positive direction. This exploratory result suggests that digital participation "
            "and general labour-market access may reflect different mechanisms."
        ),
    },
    "H4": {
        "result_assessment": "not supported",
        "short_result": (
            "Association is weakly negative and not statistically significant."
        ),
        "interpretation_for_report": (
            "Gendered part-time employment concentration is not strongly associated with the median gender wage gap "
            "indicator in this simple cross-country pairwise analysis."
        ),
    },
}

In [28]:
results_summary = hypothesis_test_results_rounded.copy()

results_summary["result_assessment"] = results_summary["hypothesis_id"].map(
    lambda h: result_interpretations[h]["result_assessment"]
)

results_summary["short_result"] = results_summary["hypothesis_id"].map(
    lambda h: result_interpretations[h]["short_result"]
)

results_summary["interpretation_for_report"] = results_summary["hypothesis_id"].map(
    lambda h: result_interpretations[h]["interpretation_for_report"]
)

summary_cols = [
    "hypothesis_id",
    "hypothesis_name",
    "status",
    "expected_direction",
    "n",
    "pearson_r",
    "pearson_p",
    "pearson_p_holm_primary",
    "pearson_holm_reject_primary",
    "spearman_rho",
    "spearman_p",
    "ols_slope",
    "ols_slope_ci_low",
    "ols_slope_ci_high",
    "ols_slope_p",
    "ols_r_squared",
    "direction_matches_expectation",
    "result_assessment",
    "short_result",
    "interpretation_for_report",
]

results_summary = results_summary[summary_cols]

results_summary

,hypothesis_id,hypothesis_name,status,expected_direction,n,pearson_r,pearson_p,pearson_p_holm_primary,pearson_holm_reject_primary,spearman_rho,spearman_p,ols_slope,ols_slope_ci_low,ols_slope_ci_high,ols_slope_p,ols_r_squared,direction_matches_expectation,result_assessment,short_result,interpretation_for_report
0,H1,STEM-to-leadership conversion,primary,positive,30,0.4269,0.0186,0.0373,True,0.2472,0.1879,0.5638,0.1015,1.0260,0.0186,0.1823,True,partial support,Positive linear association in the main sample...,There is some evidence of a positive country-l...
1,H2,STEM-to-pay equality,primary,negative,38,-0.1745,0.2949,0.2949,False,-0.1213,0.4680,-0.2113,-0.6143,0.1918,0.2949,0.0304,True,not supported / inconclusive,"Estimated association is weakly negative, as e...",Higher women's STEM graduate share is not stro...
2,H3,Digital-skills gap and labour-market participa...,secondary_exploratory,positive,32,-0.5151,0.0026,NaN,False,-0.4600,0.0081,-0.6170,-0.9998,-0.2341,0.0026,0.2653,False,expected direction not supported,Association is negative and statistically sign...,Youth coding gender gaps and overall labour-fo...
3,H4,Work-structure bottleneck,secondary,positive,36,-0.2141,0.2099,NaN,False,-0.2121,0.2143,-0.1406,-0.3642,0.0830,0.2099,0.0458,False,not supported,Association is weakly negative and not statist...,Gendered part-time employment concentration is...


In [29]:
# save
results_summary.to_csv(
    DATA_PATH + "hypothesis_results_summary_oecd.csv",
    index=False
)

## Results summary

The hypothesis testing results provide mixed evidence for the project’s expected “conversion gap” patterns.

H1 receives partial support. The primary Pearson correlation suggests a positive country-level linear association between women’s share among tertiary STEM graduates and women’s share in senior/middle management, and the corresponding OLS slope expresses this association in indicator units. However, this result is sensitive to influential observations, especially Japan. When Japan is excluded, the Pearson correlation drops substantially and is no longer statistically significant. Therefore, H1 should be interpreted as partial evidence, not as a robust general OECD-wide pattern.

H2 is not supported by strong statistical evidence. The estimated association between women’s STEM graduate share and the median gender wage gap indicator is weakly negative, which is the expected direction, but it is not statistically significant and explains little cross-country variation. This suggests that stronger women’s representation in STEM education is not, by itself, a reliable country-level marker of pay equality.

H3 does not support the expected positive direction. Instead, the association between the youth coding gender gap and the labour-force participation gender gap is negative in the main OECD sample. This negative association remains after excluding observations with pairwise year gaps greater than 2 years, although it becomes weaker in the exact-year subset. Because H3 compares different age groups and has the largest year-alignment caveat, it should remain an exploratory result.

H4 is not supported. The association between women’s relative concentration in part-time employment and the median gender wage gap indicator is weakly negative and not statistically significant. This does not establish that part-time employment is irrelevant to pay inequality; it shows only that this selected country-level bivariate relationship is not a strong standalone statistical marker in the present dataset.

Overall, the results do not support a simple linear story in which stronger STEM or digital participation indicators automatically correspond to stronger labour-market power outcomes. The strongest substantive finding is the mismatch itself: Estonia and several other OECD countries can appear relatively strong in parts of the education or participation pipeline while still showing weaker outcomes in pay equality or leadership representation.


## <a id="interpretation-and-implications-for-the-report"></a>Interpretation and implications for the report

The hypothesis testing results help define what the final report can responsibly claim and what kind of recommendation can be supported by the data.

The results do not support a simple linear story in which stronger women’s representation in STEM or digital participation automatically corresponds to stronger labour-market power outcomes. H1 provides partial evidence of a positive association between women’s STEM graduate share and women’s representation in senior/middle management, but this result is sensitive to influential observations. H2 does not provide strong evidence that higher women’s STEM graduate share is associated with lower median gender wage gap indicator values.

The exploratory H3 result also complicates a simple pipeline interpretation. The youth coding gender gap is negatively associated with the labour-force participation gap in the main OECD sample and remains negative after excluding countries with larger pairwise year gaps. Because H3 compares different age groups and has the largest year-alignment caveat, this result should not be overinterpreted. However, it does suggest that youth digital participation and overall labour-market access may reflect different country-level mechanisms.

H4 does not provide evidence that women’s relative concentration in part-time employment is strongly associated with the median gender wage gap indicator in a simple bivariate country-level analysis. This does not establish that part-time work is irrelevant to gender inequality; it shows only that the selected part-time gap is not a strong standalone marker of the wage-gap indicator in this dataset. The analysis does not measure institutional, occupational, or career-progression factors, so it cannot test whether they account for cross-country variation.

For the final report, the strongest implication is the mismatch between pipeline indicators and outcome indicators. Estonia is a relevant focal case: it has a high women’s share among tertiary STEM graduates, but it also has a high median gender wage gap indicator and below-median women’s representation in senior/middle management. This suggests that Estonia’s challenge is not only about increasing women’s participation in STEM education. It is also about understanding what happens after education: retention, career progression, pay equality, and access to leadership.

Therefore, the final recommendation should not focus narrowly on increasing the number of women in STEM education. A more data-supported recommendation would focus on the “conversion layer” between education and labour-market power. For an Estonia-facing gender-equality organization, this could mean prioritizing monitoring and interventions around women’s transition from STEM education into employment, progression into senior roles, and pay outcomes.

The final report should frame the statistical results as cross-country associations, not causal evidence. The recommendation should therefore be practical and evidence-informed, but not presented as if the tested variables prove direct causal mechanisms.
